In [10]:
#Imports
import pandas as pd
import numpy as np
from scipy.stats import f,t
#from google.colab import files

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np
from scipy.stats import t as t_dist
from sklearn.model_selection import train_test_split,  KFold

In [3]:
# Add Dataset
#uploaded = files.upload()
df_housing = pd.read_csv('Revised - Merged Housing Dataset (by ZipCode).csv', index_col = 0)
df_stab = pd.read_csv('/Users/minseokcho/Desktop/github/New DSA/NYC-DSA/datasets/final-usables/final_zip_stabilization_dataset.csv')
# df_pluto_stab = pd.read_csv('pluto_with_rent_stabilization_flag.csv')
#Data Setup

In [4]:
#Regression Function

# Figure out a way to get r2 adj higher than 0.7 but lower than 0.85
# Make second model with different variables
# Make a model correctness metric (train,test,split)

def mlr(Xk, Y):
  n = len(Y) #Number of Samples
  p = Xk.shape[1] #Number of Predictors,
  X = np.column_stack((np.ones(n),Xk)) # adds the intercept

  betahat = np.linalg.solve(X.T@X, X.T@Y)
  yhat = X @ betahat
  resid = Y - yhat

  #Sum of Squaresns
  sse = np.sum(resid**2)
  sst = np.var(Y)*(n-1)
  ssm = sst-sse

  #Mean Squares
  mse = sse/(n-p-1)
  mst = sst/(n-1)
  msm = ssm/p
  rmse = np.sqrt(mse)
  fstats = msm/mse #shows how useful each variable is at predicting


  # Variance-Covariance of Coefficients
  var_betahat = mse * np.linalg.inv(X.T @ X)
  se_betahat = np.sqrt(np.diagonal(var_betahat))

  # T-statistics for each variable
  t_stats = betahat / se_betahat

  pval = 2 * (1 - t.cdf(np.abs(t_stats), n - p - 1))

  #Model Fit
  r2 = 1 - (sse/sst) #ranges from [0,1]
  r2_adjusted = 1 - (mse/mst) #Incase of poor model fit

  # Akaike Information Criterion
  aic = n * np.log(sse / n) + 2 * (p + 1) + n * (1 + np.log(2 * np.pi))

  return r2, r2_adjusted, aic, betahat, se_betahat, pval


In [4]:
df_housing = df_housing.reset_index(drop=True)
df_housing

,geoid,zip,borough,rent_burden_total_renter_households,rent_lt_10pct_income,rent_10_to_14_9pct_income,rent_15_to_19_9pct_income,rent_20_to_24_9pct_income,rent_25_to_29_9pct_income,rent_30_to_34_9pct_income,...,built_2020_or_later,built_2010_2019,built_2000_2009,built_1990_1999,built_1980_1989,built_1970_1979,built_1960_1969,built_1950_1959,built_1940_1949,built_1939_or_earlier
0,86000US10001,10001,Manhattan,11939.0,917.0,1502.0,1801.0,1484.0,1181.0,1074.0,...,280.0,5049.0,2404.0,296.0,590.0,423.0,3290.0,553.0,508.0,4963.0
1,86000US10002,10002,Manhattan,29732.0,1688.0,2438.0,2857.0,3466.0,3197.0,3752.0,...,403.0,1940.0,1590.0,1313.0,2373.0,3566.0,4822.0,6003.0,2838.0,15848.0
2,86000US10003,10003,Manhattan,15410.0,1031.0,2255.0,1722.0,1455.0,1355.0,1147.0,...,50.0,683.0,989.0,558.0,1555.0,1560.0,4611.0,2456.0,1425.0,16526.0
3,86000US10004,10004,Manhattan,1060.0,21.0,119.0,255.0,84.0,39.0,128.0,...,0.0,221.0,421.0,44.0,115.0,18.0,33.0,5.0,6.0,1384.0
4,86000US10005,10005,Manhattan,4323.0,77.0,515.0,461.0,664.0,559.0,364.0,...,0.0,161.0,841.0,601.0,125.0,515.0,393.0,339.0,218.0,3413.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,86000US11691,11691,Queens,16665.0,1049.0,1126.0,915.0,1993.0,1893.0,1332.0,...,276.0,925.0,2531.0,1196.0,1430.0,4074.0,3541.0,4225.0,1461.0,4413.0
208,86000US11692,11692,Queens,5400.0,342.0,471.0,405.0,408.0,251.0,426.0,...,135.0,1175.0,1307.0,361.0,275.0,1368.0,1223.0,1205.0,382.0,984.0
209,86000US11693,11693,Queens,2847.0,112.0,238.0,240.0,301.0,375.0,220.0,...,0.0,377.0,811.0,406.0,71.0,495.0,1128.0,997.0,520.0,1117.0
210,86000US11694,11694,Queens,3963.0,310.0,263.0,685.0,572.0,284.0,332.0,...,44.0,260.0,1090.0,177.0,329.0,592.0,2108.0,1046.0,560.0,2872.0


In [5]:
df_stab

,zipcode,total_units,stabilized_units,stabilized_buildings,total_buildings,stabilization_share
0,10001,22303.0,12310.0,127,356,0.551944
1,10002,36840.0,14431.0,601,1180,0.391721
2,10003,33576.0,18652.0,602,1479,0.555516
3,10004,4492.0,1248.0,5,37,0.277827
4,10005,6871.0,2923.0,8,25,0.425411
...,...,...,...,...,...,...
181,11692,8348.0,83.0,16,2083,0.009943
182,11693,4727.0,340.0,48,1526,0.071927
183,11694,9553.0,1423.0,29,3513,0.148958
184,11697,2837.0,0.0,0,3,0.000000


In [5]:
# DATA PREP
df_housing['zip'] = pd.to_numeric(df_housing['zip'], errors='coerce')
df_housing = df_housing.dropna(subset=['zip'])
df_housing['zip'] = df_housing['zip'].astype(int)

#Merges data on shared components
df_merged = pd.merge(df_housing, df_stab, left_on='zip',right_on ='zipcode', how='inner')

# FEATURE ENGINEERING
# Calculate Housing Supply with Vacancy Rate
df_merged['vacancy_rate'] = df_merged['vacant_units'] / df_merged['housing_units_total']

# Homeownership Rate
df_merged['homeownership_rate'] = df_merged['owner_occupied_units'] / df_merged['housing_units_total']

# Construction Share (Everything built 2010 or later)
df_merged['new_construction_share'] = (df_merged['built_2020_or_later'] + df_merged['built_2010_2019']) / df_merged['housing_units_total']

# Renter Density (Share of units occupied by renters)
df_merged['renter_density'] = df_merged['renter_occupied_units'] / df_merged['housing_units_total']

# Pre-War Building Share (Built 1939 or earlier)
df_merged['pre_war_share'] = df_merged['built_1939_or_earlier'] / df_merged['housing_units_total']

df_merged

,geoid,zip,borough,rent_burden_total_renter_households,rent_lt_10pct_income,rent_10_to_14_9pct_income,rent_15_to_19_9pct_income,rent_20_to_24_9pct_income,rent_25_to_29_9pct_income,rent_30_to_34_9pct_income,...,total_units,stabilized_units,stabilized_buildings,total_buildings,stabilization_share,vacancy_rate,homeownership_rate,new_construction_share,renter_density,pre_war_share
0,86000US10001,10001,Manhattan,11939.0,917.0,1502.0,1801.0,1484.0,1181.0,1074.0,...,22303.0,12310.0,127,356,0.551944,0.142678,0.206908,0.290314,0.650414,0.270375
1,86000US10002,10002,Manhattan,29732.0,1688.0,2438.0,2857.0,3466.0,3197.0,3752.0,...,36840.0,14431.0,601,1180,0.391721,0.101730,0.167682,0.057573,0.730588,0.389424
2,86000US10003,10003,Manhattan,15410.0,1031.0,2255.0,1722.0,1455.0,1355.0,1147.0,...,33576.0,18652.0,602,1479,0.555516,0.194325,0.298984,0.024102,0.506691,0.543386
3,86000US10004,10004,Manhattan,1060.0,21.0,119.0,255.0,84.0,39.0,128.0,...,4492.0,1248.0,5,37,0.277827,0.170004,0.358255,0.098353,0.471740,0.615932
4,86000US10005,10005,Manhattan,4323.0,77.0,515.0,461.0,664.0,559.0,364.0,...,6871.0,2923.0,8,25,0.425411,0.205571,0.140024,0.024372,0.654405,0.516652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,86000US11691,11691,Queens,16665.0,1049.0,1126.0,915.0,1993.0,1893.0,1332.0,...,24882.0,9036.0,156,5608,0.363154,0.042913,0.264789,0.049892,0.692298,0.183325
174,86000US11692,11692,Queens,5400.0,342.0,471.0,405.0,408.0,251.0,426.0,...,8348.0,83.0,16,2083,0.009943,0.040998,0.317291,0.155674,0.641711,0.116934
175,86000US11693,11693,Queens,2847.0,112.0,238.0,240.0,301.0,375.0,220.0,...,4727.0,340.0,48,1526,0.071927,0.061297,0.457953,0.063661,0.480750,0.188619
176,86000US11694,11694,Queens,3963.0,310.0,263.0,685.0,572.0,284.0,332.0,...,9553.0,1423.0,29,3513,0.148958,0.100573,0.462877,0.033488,0.436550,0.316369


In [7]:
df_merged.columns

Index(['geoid', 'zip', 'borough', 'rent_burden_total_renter_households',
       'rent_lt_10pct_income', 'rent_10_to_14_9pct_income',
       'rent_15_to_19_9pct_income', 'rent_20_to_24_9pct_income',
       'rent_25_to_29_9pct_income', 'rent_30_to_34_9pct_income',
       'rent_35_to_39_9pct_income', 'rent_40_to_49_9pct_income',
       'rent_50pct_or_more_income', 'rent_not_computed', 'housing_units_total',
       'median_gross_rent', 'median_household_income', 'occupancy_total_units',
       'occupied_units', 'vacant_units', 'tenure_total_occupied_units',
       'owner_occupied_units', 'renter_occupied_units', 'total_population',
       'year_built_total_units', 'built_2020_or_later', 'built_2010_2019',
       'built_2000_2009', 'built_1990_1999', 'built_1980_1989',
       'built_1970_1979', 'built_1960_1969', 'built_1950_1959',
       'built_1940_1949', 'built_1939_or_earlier', 'total_units',
       'stabilized_units', 'stabilized_buildings', 'total_buildings',
       'stabilization_sha

In [8]:
# 3. FEATURE ENGINEERING
# Calculate Housing Supply with Vacancy Rate
df_merged['vacancy_rate'] = df_merged['vacant_units'] / df_merged['housing_units_total']

# 4. VARIABLE SELECTION
# Target (Y) is Median Gross Rent, X represents variables we think affect rent
cols_to_keep = ['median_gross_rent', 'stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']]
feature_names = ['Intercept', 'Stabilization Share', 'Median Income', 'Vacancy Rate', 'Total Population']

# Graph Functions

## Partial Regression Plot

In [6]:
def partial_regression_plot(Y, Xk, feature_names):
    n = len(Y)
    cols = Xk.shape[1]

    fig = go.Figure()

    for j in range(cols):
        # Other predictors (excluding j)
        X_others = np.delete(Xk, j, axis=1)
        X_others_int = np.column_stack([np.ones(n), X_others])

        # Residuals of Y ~ others
        b_Y = np.linalg.solve(X_others_int.T @ X_others_int, X_others_int.T @ Y)
        e_Y = Y - X_others_int @ b_Y

        # Residuals of Xj ~ others
        Xj = Xk[:, j]
        b_Xj = np.linalg.solve(X_others_int.T @ X_others_int, X_others_int.T @ Xj)
        e_Xj = Xj - X_others_int @ b_Xj

        # Regression line through residuals
        slope, intercept = np.polyfit(e_Xj, e_Y, 1)
        x_line = np.linspace(e_Xj.min(), e_Xj.max(), 100)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=e_Xj, y=e_Y, mode='markers',
            marker=dict(size=6, opacity=0.6),
            name='Observations'
        ))
        fig.add_trace(go.Scatter(
            x=x_line, y=slope * x_line + intercept,
            mode='lines', name='Partial Slope'
        ))
        fig.update_layout(title=f"Partial Regression: {feature_names[j+1]}")
        fig.update_xaxes(title_text=f"{feature_names[j+1]} (residualized)")
        fig.update_yaxes(title_text="Rent (residualized)")
        fig.show()

# Model 1

In [7]:
# VARIABLE SELECTION
# Y is our target(Median Gross Rent), X represents variables we think affect rent
cols_to_keep = ['median_gross_rent', 'stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']]
feature_names = ['Intercept', 'Stabilization Share', 'Median Income', 'Vacancy Rate', 'Total Population']

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.0147     | SAFE           
Stabilization Share       | 1.6463     | SAFE           
Median Income             | 1.4091     | SAFE           
Vacancy Rate              | 1.2334     | SAFE           


In [8]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.7819 (Accuracy on data it studied)
Testing R-Squared:    0.8664 (Accuracy on HIDDEN data)
Test RMSE (Error):   $248.36 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1991.3251
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 739.377977   | 0.0000     | YES         
Stabilization Share    | 140.572175   | 0.1689     | NO          
Median Income          | 0.011379     | 0.0000     | YES         
Vacancy Rate           | 1769.214451  | 0.0004     | YES         
Total Population       | 0.000535     | 0.6148     | NO          


## CV

In [11]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Stabilization, Income, Vacancy, Population)...")
print("======================================================================")

model_features = ['stabilization_share', 'median_household_income', 'vacancy_rate', 'total_population']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (Stabilization, Income, Vacancy, Population)...
Fold 1: R-Squared = 0.6869 | RMSE = $233.30
Fold 2: R-Squared = 0.6509 | RMSE = $309.03
Fold 3: R-Squared = 0.8002 | RMSE = $289.15
Fold 4: R-Squared = 0.6315 | RMSE = $417.01
Fold 5: R-Squared = 0.8945 | RMSE = $226.58
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7328 (± 0.0998)
-> Cross-Validation RMSE:      $295.01 (± $68.71)
-> FINAL Test Set R-Squared:           0.8664
-> FINAL Test Set RMSE:               $248.36



## Graph

In [12]:
# Actual vs. Predicted Plot

Y = np.atleast_1d(Y).flatten()       # flatten FIRST
n = len(Y)
X = np.column_stack((np.ones(n), Xk))
Y_pred = X @ betahat                  # now both are guaranteed 1D
residuals = Y - Y_pred

fig = go.Figure()

# Scatter: actual vs predicted
fig.add_trace(go.Scatter(
    x=Y_pred,
    y=Y,
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Observations"
))

# Perfect prediction line (45-degree)
min_val = min(Y_pred.min(), Y.min())
max_val = max(Y_pred.max(), Y.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Perfect Fit"
))

fig.update_layout(
    title="Actual vs Predicted Median Gross Rent",
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
)

fig.update_xaxes(title_text="Predicted ($)")
fig.update_yaxes(title_text="Actual ($)")
#fig.write_image("actual_vs_predicted.png")

fig.show()

In [13]:
# Residual Plot (basic)
Y = np.atleast_1d(Y).flatten()
n = len(Y)
X = np.column_stack((np.ones(n), Xk))
Y_pred = X @ betahat
residuals = Y - Y_pred

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=Y_pred,
    y=residuals,
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Residuals",
    # --- ADD THESE TWO LINES ---
    customdata=np.stack([Y, Y_pred, residuals], axis=1),
    hovertemplate=(
        "<b>Actual Rent:</b> $%{customdata[0]:,.0f}<br>"
        "<b>Fitted Value:</b> $%{customdata[1]:,.0f}<br>"
        "<b>Residual:</b> %{customdata[2]:,.2f}<br>"
        "<extra></extra>"
    )
))

fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(title="Residuals vs Fitted Values")
fig.update_xaxes(title_text="Fitted Values")
fig.update_yaxes(title_text="Residuals")
#fig.write_image("residual_plot.png")
fig.show()

In [14]:
# Residual Plot (with FigureWidget)
# Maybe not use

Y = np.atleast_1d(Y).flatten()        # flatten first
n = len(Y)
X = np.column_stack((np.ones(n), Xk)) # add intercept column, matching mlr()
Y_pred = X @ betahat                
residuals = Y - Y_pred

fig = go.FigureWidget()
fig.add_trace(go.Scatter(
    x=Y_pred,
    y=residuals,
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Residuals",
    customdata=np.stack([Y, Y_pred, residuals], axis=1),
    hovertemplate=(
        "<b>Actual Rent:</b> $%{customdata[0]:,.0f}<br>"
        "<b>Fitted Value:</b> $%{customdata[1]:,.0f}<br>"
        "<b>Residual:</b> %{customdata[2]:,.2f}<br>"
        "<extra></extra>"
    )
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(title="Residuals vs Fitted Values")
fig.update_xaxes(title_text="Fitted Values")
fig.update_yaxes(title_text="Residuals")
#fig.write_image("residual_plot.png")

#fig.show()

def on_hover(trace, points, state):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    x_val = Y_pred[idx]
    y_val = residuals[idx]

    # Draw vertical line from 0 to the residual point
    fig.update_layout(shapes=[dict(
        type="line",
        x0=x_val, x1=x_val,
        y0=0,     y1=y_val,
        line=dict(color="orange", width=2, dash="dot")
    )])

def on_unhover(trace, points, state):
    # Remove the line when not hovering
    fig.update_layout(shapes=[
        # Keep only the hline (y=0 baseline)
    ])

fig.data[0].on_hover(on_hover)
fig.data[0].on_unhover(on_unhover)

fig

FigureWidget({
    'data': [{'customdata': {'bdata': ('AAAAAABCqkD6SfdzigakQBjYIjDW7Y' ... 'AAAAB0m0DI/3xs00mhQED+52ObfnzA'),
                             'dtype': 'f8',
                             'shape': '175, 3'},
              'hovertemplate': ('<b>Actual Rent:</b> $%{customd' ... 'ta[2]:,.2f}<br><extra></extra>'),
              'marker': {'opacity': 0.6, 'size': 7},
              'mode': 'markers',
              'name': 'Residuals',
              'type': 'scatter',
              'uid': '2254ce8f-4880-4216-a9df-73c4ef6d26a7',
              'x': {'bdata': ('+kn3c4oGpEDb87qomniYQFoEPux6Aq' ... '836ryVQBGSi5mJr5lAyP98bNNJoUA='),
                    'dtype': 'f8'},
              'y': {'bdata': ('GNgiMNbtiEBsz+uiahJ1wMB0P3ii8F' ... '2+UcdswERILmYmvnrAQP7nY5t+fMA='),
                    'dtype': 'f8'}}],
    'layout': {'shapes': [{'line': {'color': 'red', 'dash': 'dash'},
                           'type': 'line',
                           'x0': 0,
                           'x1': 1,


In [16]:
# Income vs. Rent with regression line

income = df_model['median_household_income'].values
slope, intercept = np.polyfit(income, Y, 1)
x_line = np.linspace(income.min(), income.max(), 100)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=income, y=Y, mode='markers', name='Observations'))
fig2.add_trace(go.Scatter(x=x_line, y=slope*x_line+intercept, mode='lines', name='Regression Line'))
fig2.update_layout(title="Median Income vs. Median Rent")
fig2.update_xaxes(title_text="Median Income ($)")
fig2.update_yaxes(title_text="Median Rent ($)")
fig2.show()

In [17]:
# partial Regression plot
partial_regression_plot(Y, np.array(Xk), feature_names)

In [18]:
# Coefficient Plot (Forest Plot)

alpha = 0.05
df_resid = n - Xk.shape[1] - 1
t_crit = t_dist.ppf(1 - alpha/2, df_resid)

ci_lower = betahat[1:] - t_crit * se_betahat[1:]  # exclude intercept
ci_upper = betahat[1:] + t_crit * se_betahat[1:]
labels = feature_names[1:]  # exclude 'Intercept'

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=betahat[1:], y=labels,
    mode='markers', marker=dict(size=10),
    error_x=dict(
        type='data',
        symmetric=False,
        array=ci_upper - betahat[1:],
        arrayminus=betahat[1:] - ci_lower
    ),
    name='Coefficient'
))
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(title="Coefficient Plot with 95% Confidence Intervals")
fig.update_xaxes(title_text="Coefficient Value")
fig.update_yaxes(title_text="Predictor")
fig.show()

# Model 2

In [19]:
# VARIABLE SELECTION
cols_to_keep = ['median_gross_rent', 'median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']]

feature_names = ['Intercept', 'Median Income', 'Vacancy Rate', 'Homeownership Rate', 'New Construction Share']

# Multicollinearity Check
# --- NEW MATH: MULTICOLLINEARITY CHECKS ---
# 1. Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# 2. Calculate Variance Inflation Factors (VIF)
# The VIFs are precisely the diagonal elements of the inverse of the correlation matrix
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.6983     | SAFE           
Median Income             | 1.6691     | SAFE           
Vacancy Rate              | 1.4785     | SAFE           
Homeownership Rate        | 1.2385     | SAFE           


In [20]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.8012 (Accuracy on data it studied)
Testing R-Squared:    0.8985 (Accuracy on HIDDEN data)
Test RMSE (Error):   $216.50 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1978.3163
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 850.602737   | 0.0000     | YES         
Median Income          | 0.011505     | 0.0000     | YES         
Vacancy Rate           | 1283.233337  | 0.0133     | YES         
Homeownership Rate     | -246.801868  | 0.0806     | NO          
New Construction Share | 895.987908   | 0.0142     | YES         


## CV

In [21]:

# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Income, Vacancy, Homeownership, New Construction)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = ['median_household_income', 'vacancy_rate', 'homeownership_rate', 'new_construction_share']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (Income, Vacancy, Homeownership, New Construction)...
Fold 1: R-Squared = 0.7730 | RMSE = $198.64
Fold 2: R-Squared = 0.6970 | RMSE = $287.93
Fold 3: R-Squared = 0.8014 | RMSE = $288.28
Fold 4: R-Squared = 0.6754 | RMSE = $391.33
Fold 5: R-Squared = 0.9016 | RMSE = $218.83
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7697 (± 0.0807)
-> Cross-Validation RMSE:      $277.00 (± $67.59)
-> FINAL Test Set R-Squared:           0.8985
-> FINAL Test Set RMSE:               $216.50



## Graphs

In [22]:
# Actual vs. Predicted Plot

Y = np.atleast_1d(Y).flatten()       # flatten FIRST
n = len(Y)
X = np.column_stack((np.ones(n), Xk))
Y_pred = X @ betahat                  # now both are guaranteed 1D
residuals = Y - Y_pred

fig = go.Figure()

# Scatter: actual vs predicted
fig.add_trace(go.Scatter(
    x=Y_pred,
    y=Y,
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Observations"
))

# Perfect prediction line (45-degree)
min_val = min(Y_pred.min(), Y.min())
max_val = max(Y_pred.max(), Y.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Perfect Fit"
))

fig.update_layout(
    title="Actual vs Predicted Median Gross Rent",
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
)

fig.update_xaxes(title_text="Predicted ($)")
fig.update_yaxes(title_text="Actual ($)")
#fig.write_image("actual_vs_predicted.png")

fig.show()

In [23]:
# partial Regression plot
partial_regression_plot(Y, np.array(Xk), feature_names)

In [24]:
# Coefficient Plot (Forest Plot)

alpha = 0.05
df_resid = n - Xk.shape[1] - 1
t_crit = t_dist.ppf(1 - alpha/2, df_resid)

ci_lower = betahat[1:] - t_crit * se_betahat[1:]  # exclude intercept
ci_upper = betahat[1:] + t_crit * se_betahat[1:]
labels = feature_names[1:]  # exclude 'Intercept'

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=betahat[1:], y=labels,
    mode='markers', marker=dict(size=10),
    error_x=dict(
        type='data',
        symmetric=False,
        array=ci_upper - betahat[1:],
        arrayminus=betahat[1:] - ci_lower
    ),
    name='Coefficient'
))
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(title="Coefficient Plot with 95% Confidence Intervals")
fig.update_xaxes(title_text="Coefficient Value")
fig.update_yaxes(title_text="Predictor")
fig.show()

# Model 3

In [25]:
# VARIABLE SELECTION
cols_to_keep = ['median_gross_rent', 'median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[['median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']]

feature_names = ['Intercept', 'Median Income', 'Stabilization Share', 'Renter Density', 'Pre-War Share']

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")


MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.1753     | SAFE           
Median Income             | 1.5528     | SAFE           
Stabilization Share       | 1.7733     | SAFE           
Renter Density            | 1.0731     | SAFE           


In [26]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.7776 (Accuracy on data it studied)
Testing R-Squared:    0.8698 (Accuracy on HIDDEN data)
Test RMSE (Error):   $245.20 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1994.0153
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 536.367881   | 0.0000     | YES         
Median Income          | 0.013175     | 0.0000     | YES         
Stabilization Share    | -63.493498   | 0.6192     | NO          
Renter Density         | 544.035993   | 0.0016     | YES         
Pre-War Share          | -129.644368  | 0.3652     | NO          


## CV

In [27]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (Income, Stabilization, Renter Density, Pre-War)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = ['median_household_income', 'stabilization_share', 'renter_density', 'pre_war_share']

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# ---


Evaluating Model (Income, Stabilization, Renter Density, Pre-War)...
Fold 1: R-Squared = 0.7978 | RMSE = $187.47
Fold 2: R-Squared = 0.6669 | RMSE = $301.87
Fold 3: R-Squared = 0.7481 | RMSE = $324.62
Fold 4: R-Squared = 0.6117 | RMSE = $428.04
Fold 5: R-Squared = 0.8777 | RMSE = $243.95


## Graphs

In [28]:
# Actual vs. Predicted Plot

Y = np.atleast_1d(Y).flatten()       # flatten FIRST
n = len(Y)
X = np.column_stack((np.ones(n), Xk))
Y_pred = X @ betahat                  # now both are guaranteed 1D
residuals = Y - Y_pred

fig = go.Figure()

# Scatter: actual vs predicted
fig.add_trace(go.Scatter(
    x=Y_pred,
    y=Y,
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Observations"
))

# Perfect prediction line (45-degree)
min_val = min(Y_pred.min(), Y.min())
max_val = max(Y_pred.max(), Y.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Perfect Fit"
))

fig.update_layout(
    title="Actual vs Predicted Median Gross Rent",
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
)

fig.update_xaxes(title_text="Predicted ($)")
fig.update_yaxes(title_text="Actual ($)")
#fig.write_image("actual_vs_predicted.png")

fig.show()

In [29]:
# partial Regression plot
partial_regression_plot(Y, np.array(Xk), feature_names)

In [30]:
# Coefficient Plot (Forest Plot)

alpha = 0.05
df_resid = n - Xk.shape[1] - 1
t_crit = t_dist.ppf(1 - alpha/2, df_resid)

ci_lower = betahat[1:] - t_crit * se_betahat[1:]  # exclude intercept
ci_upper = betahat[1:] + t_crit * se_betahat[1:]
labels = feature_names[1:]  # exclude 'Intercept'

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=betahat[1:], y=labels,
    mode='markers', marker=dict(size=10),
    error_x=dict(
        type='data',
        symmetric=False,
        array=ci_upper - betahat[1:],
        arrayminus=betahat[1:] - ci_lower
    ),
    name='Coefficient'
))
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(title="Coefficient Plot with 95% Confidence Intervals")
fig.update_xaxes(title_text="Coefficient Value")
fig.update_yaxes(title_text="Predictor")
fig.show()

# Model 4

In [31]:
#VARIABLE SELECTION
cols_to_keep = [
    'median_gross_rent', 'median_household_income', 'stabilization_share',
    'vacancy_rate', 'homeownership_rate', 'new_construction_share', 'pre_war_share'
]
df_model = df_merged[cols_to_keep].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values
Xk = df_model[[
    'median_household_income', 'stabilization_share', 'vacancy_rate',
    'homeownership_rate', 'new_construction_share', 'pre_war_share'
]]

feature_names = [
    'Intercept', 'Median Income', 'Stabilization Share', 'Vacancy Rate',
    'Homeownership Rate', 'New Construction Share', 'Pre-War Share'
]

# Multicollinearity Check
# Calculate the Correlation Matrix of the predictors
corr_matrix = np.corrcoef(Xk.values, rowvar=False)

# Calculate Variance Inflation Factors (VIF)
inverse_corr_matrix = np.linalg.inv(corr_matrix)
vifs = np.diagonal(inverse_corr_matrix)


print("======================================================================")
print(f"MULTICOLLINEARITY DIAGNOSTICS (VIF)")
print("======================================================================")
print(f"{'Predictor Variable':<25} | {'VIF Score':<10} | {'Status':<15}")
print("-" * 65)

for i in range(len(vifs)):
    vif_val = vifs[i]
    if vif_val > 5:
        status = "DANGER (>5)"
    elif vif_val > 2.5:
        status = "MODERATE"
    else:
        status = "SAFE"
    print(f"{feature_names[i]:<25} | {vif_val:<10.4f} | {status:<15}")

MULTICOLLINEARITY DIAGNOSTICS (VIF)
Predictor Variable        | VIF Score  | Status         
-----------------------------------------------------------------
Intercept                 | 1.7696     | SAFE           
Median Income             | 1.5593     | SAFE           
Stabilization Share       | 1.7631     | SAFE           
Vacancy Rate              | 2.3214     | SAFE           
Homeownership Rate        | 1.4038     | SAFE           
New Construction Share    | 1.2207     | SAFE           


In [32]:
# MODEL TRAINING

X_train, X_test, Y_train, Y_test = train_test_split(Xk, Y, test_size=0.20, random_state=42)
r_squared_train, r2_adj_train, aic_train, betahat, se_betahat, p_values_array = mlr(X_train, Y_train)

betahat = np.atleast_1d(betahat)
p_values_array = np.atleast_1d(p_values_array)

# MODEL TESTING
n_test = len(Y_test)
X_test_intercept = np.column_stack((np.ones(n_test), X_test))

#Predictions based on test
Y_predictions = X_test_intercept @ betahat

# Calculate Accuracy
residuals = Y_test - Y_predictions
test_sse = np.sum(residuals**2)
test_sst = np.sum((Y_test - np.mean(Y_test))**2)

test_r_squared = 1 - (test_sse / test_sst)
#Average of what we were off by (in dollars)
test_rmse = np.sqrt(test_sse / n_test)

print("======================================================================")
print(f"MACHINE LEARNING EVALUATION: TRAIN VS. TEST")
print("======================================================================")
print(f"Data Split:           {len(Y_train)} Training Neighborhoods | {n_test} Testing Neighborhoods")
print(f"Training R-Squared:   {r_squared_train:.4f} (Accuracy on data it studied)")
print(f"Testing R-Squared:    {test_r_squared:.4f} (Accuracy on HIDDEN data)")
print(f"Test RMSE (Error):   ${test_rmse:.2f} (Average prediction error in dollars)")

print("\n======================================================================")
print(f"MODEL FIT SUMMARY (BASED ON TRAINING SET)")
print("======================================================================")
print(f"AIC Score:          {aic_train:.4f}")
print("======================================================================")
print(f"{'Predictor Variable':<22} | {'Coefficient':<12} | {'P-Value':<10} | {'Significant?':<12}")
print("-" * 65)

for i in range(len(feature_names)):
    sig = "YES" if p_values_array[i] < 0.05 else "NO"
    coef_str = f"{betahat[i]:.6f}"
    pval_str = f"{p_values_array[i]:.4f}"

    print(f"{feature_names[i]:<22} | {coef_str:<12} | {pval_str:<10} | {sig:<12}")
print("======================================================================")

MACHINE LEARNING EVALUATION: TRAIN VS. TEST
Data Split:           140 Training Neighborhoods | 35 Testing Neighborhoods
Training R-Squared:   0.8015 (Accuracy on data it studied)
Testing R-Squared:    0.9001 (Accuracy on HIDDEN data)
Test RMSE (Error):   $214.69 (Average prediction error in dollars)

MODEL FIT SUMMARY (BASED ON TRAINING SET)
AIC Score:          1982.1092
Predictor Variable     | Coefficient  | P-Value    | Significant?
-----------------------------------------------------------------
Intercept              | 890.635485   | 0.0000     | YES         
Median Income          | 0.011591     | 0.0000     | YES         
Stabilization Share    | -38.784397   | 0.7501     | NO          
Vacancy Rate           | 1240.660017  | 0.0202     | YES         
Homeownership Rate     | -295.303102  | 0.1042     | NO          
New Construction Share | 849.074873   | 0.0307     | YES         
Pre-War Share          | -44.295977   | 0.7613     | NO          


## CV

In [33]:
# ======================================================================
# CROSS-VALIDATION & FINAL MODEL TEST
# ======================================================================
# We split on df_model which you already cleaned in your block above
df_train, df_test = train_test_split(df_model, test_size=0.20, random_state=42)

Y_train = df_train['median_gross_rent'].values
Y_test = df_test['median_gross_rent'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n======================================================================")
print("Evaluating Model (The Master Mix: 6 Features)...")
print("======================================================================")

# Update features to match your new Variable Selection
model_features = [
    'median_household_income', 'stabilization_share', 'vacancy_rate',
    'homeownership_rate', 'new_construction_share', 'pre_war_share'
]

X_train_model = df_train[model_features].values
X_test_model = df_test[model_features].values

# 1. Run Cross-Validation on the Training Data
cv_r2_scores = []
cv_rmse_scores = []
fold_number = 1

for train_index, val_index in kf.split(X_train_model):
    X_fold_train, X_fold_val = X_train_model[train_index], X_train_model[val_index]
    Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

    # Train on 4 folds
    _, _, _, betahat_fold, _, _ = mlr(X_fold_train, Y_fold_train)

    # Validate on the 1 remaining fold
    n_val = len(Y_fold_val)
    X_fold_val_int = np.column_stack((np.ones(n_val), X_fold_val))
    fold_predictions = X_fold_val_int @ betahat_fold

    # Calculate Fold Metrics
    fold_sse = np.sum((Y_fold_val - fold_predictions)**2)
    fold_sst = np.sum((Y_fold_val - np.mean(Y_fold_val))**2)

    fold_r2 = 1 - (fold_sse / fold_sst)
    fold_rmse = np.sqrt(fold_sse / n_val)

    cv_r2_scores.append(fold_r2)
    cv_rmse_scores.append(fold_rmse)

    print(f"Fold {fold_number}: R-Squared = {fold_r2:.4f} | RMSE = ${fold_rmse:.2f}")
    fold_number += 1

# --- STANDARD DEVIATION ---
cv_r2_mean = np.mean(cv_r2_scores)
cv_r2_std = np.std(cv_r2_scores)

cv_rmse_mean = np.mean(cv_rmse_scores)
cv_rmse_std = np.std(cv_rmse_scores)

print("----------------------------------------------------------------------")
print(f"-> Cross-Validation R-Squared: {cv_r2_mean:.4f} (± {cv_r2_std:.4f})")
print(f"-> Cross-Validation RMSE:      ${cv_rmse_mean:.2f} (± ${cv_rmse_std:.2f})")
print("======================================================================")

# 2. Final Test!
_, _, _, betahat_final, _, _ = mlr(X_train_model, Y_train)

X_test_model_int = np.column_stack((np.ones(len(Y_test)), X_test_model))

final_predictions = X_test_model_int @ betahat_final
final_sse = np.sum((Y_test - final_predictions)**2)
final_sst = np.sum((Y_test - np.mean(Y_test))**2)
final_test_r2 = 1 - (final_sse / final_sst)
final_test_rmse = np.sqrt(final_sse / len(Y_test))

print(f"-> FINAL Test Set R-Squared:           {final_test_r2:.4f}")
print(f"-> FINAL Test Set RMSE:               ${final_test_rmse:.2f}\n")


Evaluating Model (The Master Mix: 6 Features)...
Fold 1: R-Squared = 0.7488 | RMSE = $208.95
Fold 2: R-Squared = 0.6980 | RMSE = $287.42
Fold 3: R-Squared = 0.7670 | RMSE = $312.25
Fold 4: R-Squared = 0.6477 | RMSE = $407.73
Fold 5: R-Squared = 0.9005 | RMSE = $220.04
----------------------------------------------------------------------
-> Cross-Validation R-Squared: 0.7524 (± 0.0850)
-> Cross-Validation RMSE:      $287.28 (± $71.82)
-> FINAL Test Set R-Squared:           0.9001
-> FINAL Test Set RMSE:               $214.69



## Graphs

In [ ]:
# Actual vs. Predicted Plot
Y = np.atleast_1d(Y).flatten()       # flatten FIRST
n = len(Y)
X = np.column_stack((np.ones(n), Xk))
Y_pred = X @ betahat                  # now both are guaranteed 1D
residuals = Y - Y_pred

fig = go.Figure()

# Scatter: actual vs predicted
fig.add_trace(go.Scatter(
    x=Y_pred,
    # X = Y_predictions
    y=Y,
    # y = Y_test
    mode="markers",
    marker=dict(size=7, opacity=0.6),
    name="Observations"
))

# Perfect prediction line (45-degree)
min_val = min(Y_pred.min(), Y.min())
max_val = max(Y_pred.max(), Y.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Perfect Fit"
))

fig.update_layout(
    title="Actual vs Predicted Median Gross Rent",
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
)

fig.update_xaxes(title_text="Predicted ($)")
fig.update_yaxes(title_text="Actual ($)")
#fig.write_image("actual_vs_predicted.png")

fig.show()

In [42]:
# partial Regression plot
partial_regression_plot(Y, np.array(Xk), feature_names)

In [36]:
# Coefficient Plot (Forest Plot)

alpha = 0.05
df_resid = n - Xk.shape[1] - 1
t_crit = t_dist.ppf(1 - alpha/2, df_resid)

ci_lower = betahat[1:] - t_crit * se_betahat[1:]  # exclude intercept
ci_upper = betahat[1:] + t_crit * se_betahat[1:]
labels = feature_names[1:]  # exclude 'Intercept'

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=betahat[1:], y=labels,
    mode='markers', marker=dict(size=10),
    error_x=dict(
        type='data',
        symmetric=False,
        array=ci_upper - betahat[1:],
        arrayminus=betahat[1:] - ci_lower
    ),
    name='Coefficient'
))
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(title="Coefficient Plot with 95% Confidence Intervals")
fig.update_xaxes(title_text="Coefficient Value")
fig.update_yaxes(title_text="Predictor")
fig.show()

# Pluto

In [28]:
pluto = pd.read_csv('/Users/minseokcho/Desktop/github/DSA_Github/lyunljl-BUDSAxNYC-SCHOOL-of-DATA_Team-A/datasets/final-usables/pluto_with_rent_stabilization_flag.csv')
pluto

/var/folders/3z/5gfy93qs7vz0g17fs_bg32xr0000gn/T/ipykernel_72179/3972068820.py:1: DtypeWarning: Columns (21,22,24,25,26,28,65,66,81,88) have mixed types. Specify dtype option on import or set low_memory=False.
  pluto = pd.read_csv('/Users/minseokcho/Desktop/github/DSA_Github/lyunljl-BUDSAxNYC-SCHOOL-of-DATA_Team-A/datasets/final-usables/pluto_with_rent_stabilization_flag.csv')


,borough,block,lot,cd,bct2020,bctcb2020,ct2010,cb2010,schooldist,council,...,plutomapid,firm07_flag,pfirm15_flag,version,dcpedited,latitude,longitude,notes,likely_rent_stabilized_binary,stabilized_units
0,QN,6173,23,411.0,4112300.0,4.112300e+10,1123.00,1002.0,26.0,19.0,...,1,NaN,NaN,25v4,NaN,40.769090,-73.772074,NaN,0,0.0
1,QN,6173,24,411.0,4112300.0,4.112300e+10,1123.00,1002.0,26.0,19.0,...,1,NaN,NaN,25v4,NaN,40.768889,-73.772150,NaN,0,0.0
2,QN,6169,26,411.0,4112300.0,4.112300e+10,1123.00,2000.0,26.0,19.0,...,1,NaN,NaN,25v4,NaN,40.767995,-73.773987,NaN,0,0.0
3,QN,6172,1,411.0,4112300.0,4.112300e+10,1123.00,2001.0,26.0,19.0,...,1,NaN,NaN,25v4,NaN,40.767116,-73.773521,NaN,0,0.0
4,QN,6172,6,411.0,4112300.0,4.112300e+10,1123.00,2001.0,26.0,19.0,...,1,NaN,NaN,25v4,NaN,40.767555,-73.773375,NaN,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
767884,BX,3101,6,206.0,2039100.0,2.039100e+10,391.00,1001.0,10.0,15.0,...,2,NaN,NaN,25v4,NaN,40.851534,-73.885552,NaN,0,0.0
767885,QN,7242,24,408.0,4127700.0,4.127700e+10,1277.00,2010.0,26.0,24.0,...,2,NaN,NaN,25v4,NaN,40.724441,-73.785204,NaN,0,0.0
767886,SI,5262,51,503.0,5015601.0,5.015601e+10,156.01,2009.0,31.0,51.0,...,1,NaN,NaN,25v4,NaN,40.543059,-74.151148,NaN,0,0.0
767887,SI,5623,68,503.0,5017011.0,5.017011e+10,170.11,1019.0,31.0,51.0,...,1,NaN,NaN,25v4,NaN,40.546046,-74.172671,NaN,0,0.0
